# Darukaa Biodiversity Baseline — Colab (v1.2.0)

**Cell-by-cell workflow.** This generalized engine supports **aquatic, terrestrial and mixed** assessments and combines Earth observation with optional field, acoustic and eDNA evidence.

The **last cell only** generates and displays the professional Year-0 HTML report.

In [ ]:
import os, sys, shutil
from pathlib import Path

os.chdir('/content')
REPO_DIR='/content/reference-benchmarking'
PKG_DIR=f'{REPO_DIR}/darukaa_adaptive_v1.0.0'

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
    !git pull --ff-only
else:
    !git clone https://github.com/G-auravSingh/reference-benchmarking.git {REPO_DIR}

os.chdir(PKG_DIR)
!pip install -e . -q --force-reinstall

for m in list(sys.modules):
    if m.startswith('darukaa_adaptive'):
        del sys.modules[m]

os.chdir('/content')
print('✓ Package synced and installed:', PKG_DIR)

## 1. Configuration

Use `mixed_lake.yaml` for the Nandoshi baseline. Switch to `aquatic_lake.yaml` or `terrestrial.yaml` when only one realm is required.

In [ ]:
from darukaa_adaptive import AssessmentConfig

CONFIG_PATH=f'{PKG_DIR}/profiles/mixed_lake.yaml'
cfg=AssessmentConfig.from_yaml(CONFIG_PATH)
print('Profile:', cfg.profile.name, cfg.profile.version)
print('Baseline:', cfg.temporal.baseline_label)
print('Validation:', cfg.validate())

## 2. Earth Engine authentication

In [ ]:
import ee

GEE_PROJECT = 'gaurav-singh-007'  # change to your authorised Earth Engine Cloud Project if needed
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print('✓ Earth Engine initialized:', GEE_PROJECT)

## 3. Upload the master site boundary

Only the **project KML/KMZ** is required. No reference KML/CSV is required by the automated reference engine.

In [ ]:
from google.colab import files

site_upload=files.upload()
site_files=[name for name in site_upload if name.lower().endswith(('.kml','.kmz'))]
if not site_files:
    raise ValueError('Please upload a .kml or .kmz site boundary')
SITE_FILE=f'/content/{site_files[0]}'
print('✓ Site:', SITE_FILE)

## 4. Optional evidence inputs

Upload any combination of:
- field metrics CSV
- acoustic metrics CSV
- structured eDNA CSV
- eDNA PDF report
- eDNA HTML report
- Krona HTML

The pipeline can score structured external metrics only when an approved comparable reference is supplied. Evidence-only records remain visible in the report.

In [ ]:
optional_upload=files.upload()
FIELD_CSV=next((f'/content/{n}' for n in optional_upload if n.lower().endswith('.csv') and 'edna' not in n.lower() and 'acoustic' not in n.lower()), None)
ACOUSTIC_CSV=next((f'/content/{n}' for n in optional_upload if 'acoustic' in n.lower() and n.lower().endswith('.csv')), None)
EDNA_CSV=next((f'/content/{n}' for n in optional_upload if 'edna' in n.lower() and n.lower().endswith('.csv')), None)
EDNA_PDF=next((f'/content/{n}' for n in optional_upload if n.lower().endswith('.pdf')), None)
EDNA_HTML=next((f'/content/{n}' for n in optional_upload if n.lower().endswith('.html') and 'krona' not in n.lower()), None)
KRONA_HTML=next((f'/content/{n}' for n in optional_upload if 'krona' in n.lower() and n.lower().endswith('.html')), None)
print({
    'FIELD_CSV':FIELD_CSV,'ACOUSTIC_CSV':ACOUSTIC_CSV,'EDNA_CSV':EDNA_CSV,
    'EDNA_PDF':EDNA_PDF,'EDNA_HTML':EDNA_HTML,'KRONA_HTML':KRONA_HTML
})

## 5. Site geometry and spatial domains

In [ ]:
from darukaa_adaptive.site import read_kml, validate_site_geometry, area_ha, make_domains

geom, parts=read_kml(SITE_FILE)
qa=validate_site_geometry(geom)
domains=make_domains(geom, cfg.spatial.riparian_buffer_m, cfg.spatial.context_buffer_km, cfg.spatial.littoral_band_m)
print('Geometry QA:', qa)
print('Parts:', list(parts)[:10])

## 6. Run the generalized baseline pipeline

This executes aquatic and terrestrial EO modules according to the selected profile, resolves the automatic regional reference framework, ingests optional field/acoustic/eDNA data, performs QA, and calculates pillar/SoN outputs where coverage permits.

In [ ]:
from darukaa_adaptive.pipeline import AssessmentPipeline

if os.path.exists(cfg.output_dir):
    shutil.rmtree(cfg.output_dir)

pipeline=AssessmentPipeline(cfg)
result=pipeline.run(
    SITE_FILE,
    field_csv=FIELD_CSV,
    acoustic_csv=ACOUSTIC_CSV,
    edna_csv=EDNA_CSV,
    edna_pdf=EDNA_PDF,
    edna_html=EDNA_HTML,
    krona_html=KRONA_HTML,
)
print('✓ Pipeline completed')

## 7. QA/QC review

In [ ]:
print('Automated metric QA')
display(result['qa'])

print('Reference diagnostics')
print(result['reference_diagnostics'])

## 8. Water-season series (aquatic runs)

In [ ]:
import pandas as pd

water_df=pd.DataFrame(result.get('water_periods', []))
display(water_df)

## 9. Indicator → reference → intactness → concern

In [ ]:
display(result['metric_concern'])

## 10. Pillar and State of Nature aggregation

In [ ]:
print('Pillar scorecard')
display(result['pillars'])
print('Overall')
print(result['overall'])

## 11. Readiness and evidence coverage

In [ ]:
print(result['readiness'])

## 12. eDNA evidence

Structured eDNA metrics, when supplied, are kept separate from direct EO metrics and retain their evidence class. Source PDF/HTML/Krona artefacts are copied into the final report output folder.

In [ ]:
if result.get('edna'):
    display(pd.DataFrame([e.to_dict() for e in result['edna']]))
else:
    print('No structured eDNA CSV supplied in this run.')

## 13. Final deliverable

**This is the final cell.** It generates the professional Year-0 HTML baseline report from the pipeline outputs and displays it inline.

In [ ]:
from IPython.display import IFrame, display
from darukaa_adaptive.html_report import build_html_report

REPORT_PATH=f"{cfg.output_dir}/year0_biodiversity_baseline.html"
build_html_report(result, REPORT_PATH)
print('✓ Final report:', REPORT_PATH)
display(IFrame(src=REPORT_PATH, width='100%', height=1100))